# SEPNet Evaluation — Kvasir-SEG & LDPolypDB
Loads pretrained **SEPNet** (Wang et al., IEEE TCSVT 2024) weights and computes Dice / mIoU on Kvasir-SEG and LDPolypDB.
All dataset and weight paths are discovered dynamically from `/kaggle/input/` — **no hardcoded paths**.

**Before running:** attach the Kvasir-SEG dataset, the LDPolypDB dataset, and a Kaggle Dataset/notebook output containing the `.pth` SEPNet checkpoint as inputs, and enable a GPU accelerator.

In [2]:
# Example for running a PyTorch repo in Kaggle
!pip install -r requirements.txt
# Or install specific missing dependencies:
!pip install timm==0.4.12 opencv-python scikit-learn

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 7.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalled timm-1.0.26


In [3]:
# ==== Cell 1: Environment Setup & GPU Check ====
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import timm
    from timm.models.layers import DropPath  # noqa: F401 - SEPNet backbone needs the old timm.models.layers API
except Exception:
    pip_install(["timm==0.5.4"])
    import timm

import torch
print(f"PyTorch: {torch.__version__} | timm: {timm.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = torch.device("cuda")
else:
    print("WARNING: No GPU detected — enable a GPU accelerator in Kaggle settings for reasonable runtime.")
    DEVICE = torch.device("cpu")

PyTorch: 2.10.0+cu128 | timm: 0.4.12
CUDA available: True
GPU: Tesla T4


In [4]:
# ==== Cell 2: Repository Cloning & Imports ====
REPO_DIR = "/kaggle/working/SEPNet"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/wangtong627/SEPNet.git", REPO_DIR], check=True)
else:
    print("SEPNet repo already present, skipping clone.")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import glob
from pathlib import Path

import numpy as np
import cv2
from PIL import Image
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from lib.model_for_eval import SEPNet_EvalModel  # noqa: E402

print("SEPNet repo imports OK.")

Cloning into '/kaggle/working/SEPNet'...


SEPNet repo imports OK.


In [5]:
# ==== Cell 3: Fast & Targeted Discovery for _og Datasets ====
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

# Explicit mapping targeting your input tree structure
DATASET_PATHS = {
    "HyperKvasir": {
        "root_folder": "/kaggle/input/datasets/varshasrao/hyperkvasir-og",
        "img_subpath": "/kaggle/input/datasets/varshasrao/hyperkvasir-og/segmented-images/images",
        "mask_subpath": "/kaggle/input/datasets/varshasrao/hyperkvasir-og/segmented-images/masks"
    },
    "LDPolypVideo": {
        "root_folder": "/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset",
        "img_subpath": "/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Images",  # matches Images or images inside nested zip extractions
        "mask_subpath": "/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Annotations"  # matches Annotations or masks
    },
    "Piccolo": {
        "root_folder": "/kaggle/input/datasets/varshasrao/piccolo-og",
        "img_subpath": "/kaggle/input/datasets/varshasrao/piccolo-og/Piccolo dataset-release0.1/test/polyps",
        "mask_subpath": "/kaggle/input/datasets/varshasrao/piccolo-og/Piccolo dataset-release0.1/test/masks"
    },
    "PolypDB": {
        "root_folder": "/kaggle/input/datasets/varshasrao/polypdb-og",
        "img_subpath": "/kaggle/input/datasets/varshasrao/polypdb-og/PolypDB/PolypDB/PolypDB_modality_wise/WLI/images",
        "mask_subpath": "/kaggle/input/datasets/varshasrao/polypdb-og/PolypDB/PolypDB/PolypDB_modality_wise/WLI/masks"
    }
}

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif"}
dataset_pairs = {}

for ds_name, cfg in DATASET_PATHS.items():
    root_dir = KAGGLE_INPUT / cfg["root_folder"]
    
    if not root_dir.exists():
        print(f"[{ds_name}] Root folder '{cfg['root_folder']}' not found in /kaggle/input")
        continue

    # Locate image and mask directories
    img_dirs = list(root_dir.glob(cfg["img_subpath"])) if "**" in cfg["img_subpath"] else [root_dir / cfg["img_subpath"]]
    mask_dirs = list(root_dir.glob(cfg["mask_subpath"])) if "**" in cfg["mask_subpath"] else [root_dir / cfg["mask_subpath"]]

    img_dir = img_dirs[0] if img_dirs and img_dirs[0].exists() else None
    mask_dir = mask_dirs[0] if mask_dirs and mask_dirs[0].exists() else None

    # Fallback search if exact subpath structure differs inside the zip extract
    if not img_dir or not mask_dir:
        all_dirs = [d for d in root_dir.rglob("*") if d.is_dir()]
        for d in all_dirs:
            d_name = d.name.lower()
            if not img_dir and d_name in ["images", "polyps", "img"]:
                img_dir = d
            elif not mask_dir and d_name in ["masks", "annotations", "gt"]:
                mask_dir = d

    if img_dir and mask_dir:
        img_files = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in valid_exts])
        mask_files = sorted([f for f in mask_dir.iterdir() if f.suffix.lower() in valid_exts])

        # Match images and masks by filename stem (ignoring extensions)
        mask_map = {f.stem.lower(): f for f in mask_files}
        pairs = [(img_f, mask_map[img_f.stem.lower()]) for img_f in img_files if img_f.stem.lower() in mask_map]

        if pairs:
            dataset_pairs[ds_name] = pairs
            print(f"[{ds_name}] Matched {len(pairs)} image/mask pairs from '{cfg['root_folder']}'.")
        else:
            print(f"[{ds_name}] Found folders but no filename stem matches between '{img_dir}' and '{mask_dir}'.")
    else:
        print(f"[{ds_name}] Could not resolve image/mask folders inside '{cfg['root_folder']}'.")

[HyperKvasir] Matched 1000 image/mask pairs from '/kaggle/input/datasets/varshasrao/hyperkvasir-og'.
[LDPolypVideo] Found folders but no filename stem matches between '/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Images' and '/kaggle/input/datasets/gokulraj324/ldpolypvideo-og-dataset/LDPolypVideo-20260905T141513Z-1-001/LDPolypVideo/Test/Test/Annotations'.
[Piccolo] Found folders but no filename stem matches between '/kaggle/input/datasets/varshasrao/piccolo-og/Piccolo dataset-release0.1/test/polyps' and '/kaggle/input/datasets/varshasrao/piccolo-og/Piccolo dataset-release0.1/test/masks'.
[PolypDB] Matched 3588 image/mask pairs from '/kaggle/input/datasets/varshasrao/polypdb-og'.


In [9]:
# ==== Cell 3: Robust Discovery for All Datasets ====
import re
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

DATASET_PATHS = {
    "HyperKvasir": {
        "root_folder": "hyperkvasir-og",
        "img_subpath": "segmented-images/images",
        "mask_subpath": "segmented-images/masks",
    },
    "LDPolypVideo": {
        "root_folder": "ldpolypvideo-og-dataset",
        "img_subpath": "**/Test/**/Images",
        "mask_subpath": "**/Test/**/Annotations",
    },
    "Piccolo": {
        "root_folder": "piccolo-og",
        "img_subpath": "**/test/polyps",
        "mask_subpath": "**/test/masks",
    },
    "PolypDB": {
        "root_folder": "polypdb-og",
        "img_subpath": "**/WLI/images",
        "mask_subpath": "**/WLI/masks",
    },
}

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif"}


def clean_stem(stem_str):
    """Normalizes stem names by stripping common mask indicators and suffix tags."""
    s = stem_str.lower()
    # Strip common mask tags
    for extra in ["_corrected", "corrected_", "_mask", "mask_", "_gt", "gt_", "-mask", "mask-"]:
        s = s.replace(extra, "")
    # Standardize pure numeric stems (e.g., '1' -> '0001') for robust matching
    if s.isdigit():
        s = s.zfill(4)
    return s


dataset_pairs = {}

for ds_name, cfg in DATASET_PATHS.items():
    matched_roots = [
        p for p in KAGGLE_INPUT.rglob("*") if p.is_dir() and cfg["root_folder"].lower() in p.name.lower()
    ]
    if not matched_roots:
        print(f"[{ds_name}] Root dataset folder '{cfg['root_folder']}' not found.")
        continue

    root_dir = matched_roots[0]

    # Resolve image and mask directories
    img_dirs = list(root_dir.glob(cfg["img_subpath"].lstrip("/")))
    mask_dirs = list(root_dir.glob(cfg["mask_subpath"].lstrip("/")))

    # Fallback to broad search if strict subpath isn't found
    if not img_dirs or not mask_dirs:
        img_dirs = [d for d in root_dir.rglob("*") if d.is_dir() and d.name.lower() in ["images", "polyps", "img"]]
        mask_dirs = [
            d for d in root_dir.rglob("*") if d.is_dir() and d.name.lower() in ["masks", "annotations", "gt"]
        ]

    img_dir = img_dirs[0] if img_dirs else None
    mask_dir = mask_dirs[0] if mask_dirs else None

    if img_dir and mask_dir:
        img_files = sorted([f for f in img_dir.rglob("*") if f.suffix.lower() in valid_exts])
        mask_files = sorted([f for f in mask_dir.rglob("*") if f.suffix.lower() in valid_exts])

        # Build clean-stem to path lookup
        mask_map = {clean_stem(f.stem): f for f in mask_files}

        pairs = []
        for img_f in img_files:
            c_stem = clean_stem(img_f.stem)
            if c_stem in mask_map:
                pairs.append((img_f, mask_map[c_stem]))

        if pairs:
            dataset_pairs[ds_name] = pairs
            print(f"[{ds_name}] Matched {len(pairs)} image/mask pairs.")
        else:
            sample_imgs = [f.stem for f in img_files[:3]]
            sample_masks = [f.stem for f in mask_files[:3]]
            print(f"[{ds_name}] No stem matches found.\n  Images: {sample_imgs}\n  Masks: {sample_masks}")
    else:
        print(f"[{ds_name}] Could not locate image/mask folders under '{root_dir}'.")

[HyperKvasir] Matched 1000 image/mask pairs.
[LDPolypVideo] No stem matches found.
  Images: ['0001', '0002', '0003']
  Masks: []
[Piccolo] Matched 333 image/mask pairs.
[PolypDB] Matched 3588 image/mask pairs.


In [7]:
# ==== Cell 4: SEPNet Model Initialization & Pretrained Weights Loading ====
BACKBONE_NAME = "pvt-v2-b2"
MID_CHANNELS = 64

model = SEPNet_EvalModel(
    backbone_name=BACKBONE_NAME,
    pretrained_backbone_path=None,
    mid_channels=MID_CHANNELS,
    rf_reduction_scale=4,
    rf_se_channel_reduction=16,
    df_channel_reduction=[4, 4],
    sc_hidden_scale=32,
    sc_ksize=[3, 3, 3],
    sc_gas_ksize=1,
    sc_gas_ksize_scale=[15, 15, 15],
    sc_middle_channle_scale=2,
).to(DEVICE)

def find_weight_file():
    search_roots = [KAGGLE_INPUT, Path(REPO_DIR)]
    candidates = []
    for root in search_roots:
        if root.exists():
            candidates += list(root.rglob("*.pth")) + list(root.rglob("*.pt"))
    if not candidates:
        return None
    preferred = [c for c in candidates if "sepnet" in c.name.lower() or "sep" in c.name.lower()]
    return (preferred or candidates)[0]

weight_path = find_weight_file()
print(f"Loading weights from: {weight_path}")

if weight_path and weight_path.exists():
    checkpoint = torch.load(weight_path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]
    
    # Strip module prefix if wrapped in DataParallel
    state_dict = {k.replace("module.", ""): v for k, v in checkpoint.items()}
    model.load_state_dict(state_dict, strict=False)
    print("Successfully loaded weights into SEPNet!")
else:
    raise FileNotFoundError("No valid .pth checkpoint found in /kaggle/input.")

>>> Using PVT-v2-b2 as backbone.
>>> Backbone do not use pretrained params!
Loading weights from: /kaggle/input/datasets/technogravures/sepweight/Weight_of_SEPNet.pth
Successfully loaded weights into SEPNet!


In [8]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

EPS = 1e-6
model.eval()

all_dataset_summaries = []

with torch.no_grad():
    for ds_name, pairs in dataset_pairs.items():
        print(f"\n--- Evaluating {ds_name} ---")
        per_image_results = []
        
        for img_path, mask_path in pairs:
            # 1. Load image and mask
            img = Image.open(img_path).convert("RGB")
            gt = Image.open(mask_path).convert("L")
            orig_size = gt.size[::-1] # (H, W)
            
            # Preprocessing (matches standard evaluation pipeline: 352x352 input)
            transform = T.Compose([
                T.Resize((352, 352)),
                T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
            
            img_tensor = transform(img).unsqueeze(0).to(DEVICE)
            
            # 2. Inference
            out = model(img_tensor)
            pred = out[0] if isinstance(out, (tuple, list)) else out
            
            # Interpolate to ground truth size
            pred = F.interpolate(pred, size=orig_size, mode="bilinear", align_corners=False)
            pred_mask = (torch.sigmoid(pred).squeeze().cpu().numpy() > 0.5).astype(np.float32)
            gt_mask = (np.array(gt) > 128).astype(np.float32)
            
            # 3. Compute Per-Image Metrics
            intersection = (pred_mask * gt_mask).sum()
            union = pred_mask.sum() + gt_mask.sum() - intersection
            
            dice = (2.0 * intersection + EPS) / (pred_mask.sum() + gt_mask.sum() + EPS)
            iou = (intersection + EPS) / (union + EPS)
            
            per_image_results.append({
                "image_name": img_path.name,
                "image_path": str(img_path),
                "dice": dice,
                "iou": iou
            })
        
        # Save per-image metrics to CSV
        df = pd.DataFrame(per_image_results)
        csv_filename = f"/kaggle/working/{ds_name}_per_image_metrics.csv"
        df.to_csv(csv_filename, index=False)
        print(f"Saved per-image metrics to: {csv_filename}")
        
        # Summary statistics for dataset
        mean_dice = df["dice"].mean()
        mean_iou = df["iou"].mean()
        print(f"[{ds_name} Summary] Mean Dice: {mean_dice:.4f} | Mean IoU: {mean_iou:.4f}")


--- Evaluating HyperKvasir ---
Saved per-image metrics to: /kaggle/working/HyperKvasir_per_image_metrics.csv
[HyperKvasir Summary] Mean Dice: 0.9693 | Mean IoU: 0.9426

--- Evaluating Piccolo ---
Saved per-image metrics to: /kaggle/working/Piccolo_per_image_metrics.csv
[Piccolo Summary] Mean Dice: 0.8540 | Mean IoU: 0.7940

--- Evaluating PolypDB ---
Saved per-image metrics to: /kaggle/working/PolypDB_per_image_metrics.csv
[PolypDB Summary] Mean Dice: 0.8793 | Mean IoU: 0.8145
